In [86]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [87]:
#doc du lieu 3 file train, test, submission
missing_values = ["NaN "]
df = pd.read_csv("../dataset/foodDeli/train.csv", na_values=missing_values)
test_df = pd.read_csv("../dataset/foodDeli/test.csv", na_values=missing_values)
submission_df = pd.read_csv("../dataset/foodDeli/Sample_Submission.csv")
# df.head(5)

In [88]:
#test & submission xử lý ghép cột
test_df['ID'] = test_df['ID'].astype(str).str.strip()
submission_df['ID'] = submission_df['ID'].astype(str).str.strip()

test_df.dropna(inplace=True)
test_df['Weather_conditions'] = test_df['Weather_conditions'].str.replace('conditions ', '', regex=True)
test_df = pd.merge(test_df, submission_df, on='ID', how='left')

print(test_df.columns)

Index(['ID', 'Delivery_person_ID', 'Delivery_person_Age',
       'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 'Order_Date', 'Time_Orderd',
       'Time_Order_picked', 'Weather_conditions', 'Road_traffic_density',
       'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle',
       'multiple_deliveries', 'Festival', 'City', 'Time_taken'],
      dtype='str')


In [89]:

#xóa khoảng trắng đầu cuối cho train
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
df.columns = df.columns.str.strip()

df.dropna(inplace=True)

df['Weather_conditions'] = df['Weather_conditions'].str.replace('conditions ', '', regex=True)
df.rename(columns={'Time_taken(min)': 'Time_taken'}, inplace=True)
#ép kiểu dữ liệu về số nguyên
df['Time_taken'] = df['Time_taken'].astype(str).str.replace('(min)', '', regex=False).str.strip()
#chuyển qua lại kiểu số
df['Time_taken'] = pd.to_numeric(df['Time_taken'])

## chuẩn hóa và one-hot

In [90]:

# TÍNH KHOẢNG CÁCH GIAO HÀNG (Haversine)
def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1)/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2.0)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df['Distance_km'] = haversine_distance(
    df['Restaurant_latitude'], df['Restaurant_longitude'],
    df['Delivery_location_latitude'], df['Delivery_location_longitude'])

# TÍNH THỜI GIAN CHUẨN BỊ (Phút) TRAIN
df['Order_Time_Full'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Time_Orderd'])
df['Picked_Time_Full'] = pd.to_datetime(df['Order_Date'] + ' ' + df['Time_Order_picked'])

df['Preparation_Time'] = (df['Picked_Time_Full'] - df['Order_Time_Full']).dt.total_seconds() / 60.0
df['Preparation_Time'] = np.where(df['Preparation_Time'] < 0, df['Preparation_Time'] + 1440, df['Preparation_Time'])


#TEST
test_df['Distance_km'] = haversine_distance(
    test_df['Restaurant_latitude'], test_df['Restaurant_longitude'],
    test_df['Delivery_location_latitude'], test_df['Delivery_location_longitude'])

# Tính Thời gian chuẩn bị
test_df['Order_Time_Full'] = pd.to_datetime(test_df['Order_Date'] + ' ' + test_df['Time_Orderd'])
test_df['Picked_Time_Full'] = pd.to_datetime(test_df['Order_Date'] + ' ' + test_df['Time_Order_picked'])

test_df['Preparation_Time'] = (test_df['Picked_Time_Full'] - test_df['Order_Time_Full']).dt.total_seconds() / 60.0
test_df['Preparation_Time'] = np.where(test_df['Preparation_Time'] < 0, test_df['Preparation_Time'] + 1440, test_df['Preparation_Time'])

#  MÃ HÓA ONE-HOT
categorical_cols = ['Weather_conditions', 'Road_traffic_density', 'Type_of_order', 'Type_of_vehicle', 'Festival', 'City']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)


for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

for col in test_df.columns:
    if test_df[col].dtype == 'bool':
        test_df[col] = test_df[col].astype(int)



# Xóa các cột thô không còn dùng đến
cols_to_drop = [
    'Restaurant_latitude', 'Restaurant_longitude',
    'Delivery_location_latitude', 'Delivery_location_longitude',
    'Order_Date', 'Time_Orderd', 'Time_Order_picked',
    'Order_Time_Full', 'Picked_Time_Full', 'multiple_deliveries', 'Delivery_person_ID']

df.drop(columns=cols_to_drop, errors='ignore', inplace=True)
test_df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

In [91]:

# 1. TÁCH BIẾN MỤC TIÊU (y) VÀ ĐẦU VÀO (X) cho TRAIN
y = df['Time_taken']
X = df.drop(columns=['Time_taken'])

scaler = StandardScaler()

# Gom cả 4 cột số vào đây để fit_transform 1 lần duy nhất
cols_to_scale = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Distance_km', 'Preparation_Time']

X[cols_to_scale] = scaler.fit_transform(X[cols_to_scale])

display(X.head())


,ID,Delivery_person_Age,Delivery_person_Ratings,Vehicle_condition,Distance_km,Preparation_Time,Weather_conditions_Fog,Weather_conditions_Sandstorms,Weather_conditions_Stormy,Weather_conditions_Sunny,...,Road_traffic_density_Low,Road_traffic_density_Medium,Type_of_order_Drinks,Type_of_order_Meal,Type_of_order_Snack,Type_of_vehicle_motorcycle,Type_of_vehicle_scooter,Festival_Yes,City_Semi-Urban,City_Urban
0,0x4607,1.282075,0.844658,2,-0.079949,1.226310,0,0,0,1,...,0,0,0,0,1,1,0,0,0,1
1,0xb379,0.761611,-0.421740,2,-0.022520,-1.221045,0,0,1,0,...,0,0,0,0,1,0,1,0,0,0
2,0x5d6d,-1.146759,-0.738339,0,-0.084877,1.226310,0,1,0,0,...,1,0,1,0,0,1,0,0,0,1
3,0x7a6a,1.455563,0.211459,0,-0.063999,0.002633,0,0,0,1,...,0,1,0,0,0,1,0,0,0,0
4,0x70a2,0.414634,-0.105140,1,-0.069289,1.226310,0,0,0,0,...,0,0,0,0,1,0,1,0,0,0


## chia tập validate

In [92]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_data = pd.concat([X_train, y_train], axis=1)
val_data = pd.concat([X_val, y_val], axis=1)

In [94]:
y_test = test_df['Time_taken']
X_test = test_df.drop(columns=['Time_taken'])

X_train_cols = X_train.columns
X_test = X_test.reindex(columns=X_train_cols, fill_value=0)

X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [95]:
train_file = "../dataset/foodDeli_processed/train_processed.csv"
train_data.to_csv(train_file, index=False, encoding='utf-8-sig')

val_file = "../dataset/foodDeli_processed/val_processed.csv"
val_data.to_csv(val_file, index=False, encoding='utf-8-sig')

test_file = "../dataset/foodDeli_processed/test_processed.csv"
test_df.to_csv(test_file, index=False, encoding='utf-8-sig')

print(f"Kích thước file Train: {train_data.shape}")
print(f"Kích thước file Validation: {val_data.shape}")
print(f"Kích thước file Test: {test_df.shape}")

Kích thước file Train: (33094, 23)
Kích thước file Validation: (8274, 23)
Kích thước file Test: (10291, 23)
